This notebook contains all of my work and code for Question 2 of HW5. 

2. [40 points]
Create an animation of play *2735* from game *2022100210*.  Store the animation as a *.gif* file. 
Note that there is plenty of public code for making these animations.  Cite any sources that you use.

Plan.

1.) Gather the play data from the hpc and locally store it

2.) Filter the data for this play

3.) Get gif code 

4.) Use gif code with the data for the play.

5.) Store created GIF as a .gif file in GitHub

# HPC Code:

**Comment:** get back to home directory from projects directory

cd ~

**Comment:** Looking around through projects via point and click, I found that game 2022100210 is in the tracking_week_4.csv. So I copied it over to my home directory:

cp /projects/class/spoa4001_u01/SportsTrackingTransformer/data/BigDataBowl_2024/tracking_week_4.csv tracking_week_4.csv

**Comment:** Next, I exited the hpc session/ remote and used the following code in the powershell so that I could download the tacking data locally.

scp "student:~/tracking_week_4.csv" C:\Users\ajhay\Downloads\

**Comment:** Now that the data is local, we can now create the gif in this notebook.

# **GIF Code:**

In [ ]:
!apt-get -qq update
!apt install -y -qq imagemagick 

'apt-get' is not recognized as an internal or external command,
operable program or batch file.
'apt' is not recognized as an internal or external command,
operable program or batch file.


In [1]:
# import libraries
import pandas as pd 
import seaborn as sns 
import matplotlib.pyplot as plt 
plt.rcParams['animation.html'] = 'jshtml' # Added this to avoid error during runtime

# for mpl animation
import matplotlib.animation as animation
from matplotlib import rc
rc('animation', html='html5')

In [2]:
# Import the tracking data
data = pd.read_csv(r'data/tracking_week_4.csv')

## Copy the code from kaggle

Below is all of the gif creation code that I gathered from Nick wan on Kaggle. 

Here is the link: https://www.kaggle.com/code/nickwan/animated-gif-for-plays-python



In [ ]:
# Take a look at the columns in the data 
data.columns

Index(['gameId', 'playId', 'nflId', 'displayName', 'frameId', 'time',
       'jerseyNumber', 'club', 'playDirection', 'x', 'y', 's', 'a', 'dis', 'o',
       'dir', 'event'],
      dtype='str')

In [ ]:
# The function from kaggle. 
# I had to change all 'team' occurances in the function to 'club' as thats what its called in the data.
def get_play_by_frame(fid, ax, los, one_play):
  """
  take one frame from one play, plot a scatter plot image  

  inputs:
    fid: frame ID  
    ax: current matplotlib ax  
    los: line of scrimmage (for aesthetics)  
    one_play: pandas dataframe for one play  

  output:
    seaborn axis level scatter plot  
  """
  # clear current axis (or else you'll have a tracer effect)
  ax.cla()

  # get game and play IDs
  gid = one_play['gameId'].unique()[0]
  pid = one_play['playId'].unique()[0]

  # isolates a given frame within one play
  one_frame = one_play.loc[one_play['frameId']==fid]

  # create a scatter plot, hard coded dot size to 100 
  fig1 = sns.scatterplot(x='x',y='y',data=one_frame, 
                         hue='club', ax=ax, s=100)
  
  # plots line of scrimmage 
  fig1.axvline(los, c='k', ls=':')

  # plots a simple end zone 
  fig1.axvline(0, c='k', ls='-')
  fig1.axvline(100, c='k', ls='-')

  # game and play IDs as the title
  fig1.set_title(f"game {gid} play {pid}")

  # takes out the legend (if you leave this, you'll get an annoying legend)
  fig1.legend([]).set_visible(False)

  # takes out the left, top, and right borders on the graph 
  sns.despine(left=True)

  # no y axis label
  fig1.set_ylabel('')

  # no y axis tick marks
  fig1.set_yticks([])

  # set the x and y graph limits to the entire field (from kaggle BDB page)
  fig1.set_xlim(-10,110)    
  fig1.set_ylim(0,54) 

def animate_play(one_play):    
  """
  animate a given NFL play from the BDB  

  inputs: 
    one_play: one play from the BDB data. you will want to 
      filter your dataset using gameId and playId.

  output: 
    animated gif, saved to your current working directory 

  """
  # get game and play IDs
  gid = one_play['gameId'].unique()[0]
  pid = one_play['playId'].unique()[0]

  # get line of scrimmage info from the football X location from the  first frame of data 
  los = one_play.loc[(one_play['frameId']==1) & (one_play['club']=='football'), 'x'].values[0]

  # set figure size; this is hard coded but seemed to work well  
  fig = plt.figure(figsize=(14.4, 6.4))

  # get current axis of the figure
  ax = fig.gca()

  # matplotlib animate function
  # relies on get_play_by_frame()  
  # `interval = 100` is something like frames per second i think 
  # repeat=True is to have the animation continuously repeat  
  ani = animation.FuncAnimation(fig, get_play_by_frame, 
                                frames=one_play['frameId'].unique().shape[0],
                                interval=100, repeat=True, 
                                fargs=(ax,los,one_play,))
  
  # close the matplotlib figure when done (if you're batch processing gifs, this allows you to end one gif and begin another gif of a play)
  plt.close()

  # save the matplotlib animation as a gif
  # requires imagemagick or some sort of gif renderer
  # this works in google colab if you apt install imagemagick
  ani.save(f'{gid}_{pid}.gif', writer='pillow', fps=10)
  return ani

In [ ]:
# Set a gameId and a playId
gameId = 2022100210
playId = 2735

In [ ]:
# store the play data and examine its shape
play = data.loc[(data['gameId']==gameId) & (data['playId']==playId)]
play.shape

(943, 17)

In [ ]:
# Create the gif animation of the play
ani = animate_play(play)

<Figure size 640x480 with 0 Axes>

The code creates the gif file that is here in the main branch of the GitHub. 

The orange dots are the offense, the blue dots are the defense, and the green dot is the football.